In [ ]:
import sys
from pathlib import Path
import os

sys.path.append("..")

In [ ]:
import importlib
import dataset.load_dataset as load_dataset

In [ ]:
importlib.reload(load_dataset)
from dataset.load_dataset import SROIEDataset

# load image imports
from PIL import Image
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as patches

In [ ]:
# Detect if running in Google Colab
IN_COLAB = "google.colab" in str(get_ipython())
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')
    DATASET_DIR = Path("/content/drive/MyDrive/datasets/my_dataset")
else:
    # "/Users/yourname/datasets/my_dataset" should be replaced with the actual path to your dataset on your local machine
    DATASET_DIR = Path("/Users/yourname/datasets/my_dataset")
print("Dataset path:", DATASET_DIR)

In [ ]:
data_splits_path = DATASET_DIR / "versions/1"

In [ ]:
sroie_dataset = SROIEDataset(dataset_dir=data_splits_path, debug=False)

In [ ]:
sroie_dataset.train.head()

In [ ]:
sample = sroie_dataset.train.iloc[100]
sample

In [ ]:
image = Image.open(sample["img_path"])
plt.figure(figsize=(12, 12))
plt.imshow(image)
plt.axis("off");

In [ ]:
def read_box_file(path):
    rows = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            parts = line.strip().split(",")
            coords = list(map(int, parts[:8]))
            text = ",".join(parts[8:])
            rows.append({
                "x1": coords[0],
                "y1": coords[1],
                "x2": coords[2],
                "y2": coords[3],
                "x3": coords[4],
                "y3": coords[5],
                "x4": coords[6],
                "y4": coords[7],
                "text": text
            })
    return pd.DataFrame(rows)

def draw_boxes(ocr_df: pd.DataFrame, image: Image.Image):
    fig, ax = plt.subplots(figsize=(14, 14))
    ax.imshow(image)
    for _, row in ocr_df.iterrows():
        x = row["x1"]
        y = row["y1"]
        width = row["x2"] - row["x1"]
        height = row["y4"] - row["y1"]
        rect = patches.Rectangle(
            (x, y),
            width,
            height,
            linewidth=1,
            edgecolor="red",
            facecolor="none"
        )
        ax.add_patch(rect)
        ax.text(
            x,
            y - 2,
            row["text"][:15],
            fontsize=6,
            color="blue"
        )
    plt.axis("off")
    plt.show()

In [ ]:
ocr_df = read_box_file(sample["box_path"])
#ocr_df.head()
draw_boxes(ocr_df, image)

In [ ]:
with open(sample["ent_path"], "r", encoding="utf-8") as f:
    labels = f.read()

print(labels)